In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware


agent= create_agent(
    model= "gpt-5-nano",
    checkpointer= InMemorySaver(),
    middleware= [
        SummarizationMiddleware(
        model= "gpt-4o-mini",
        trigger= ("tokens", 100),
        keep= ("messages", 1)
    )
    ]
)

In [4]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response= agent.invoke( 
{
    "messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?")
    ]
},
{
    "configurable": {
        "thread_id": "1"
    }
}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about fictional details related to Lunapolis, the capital of the moon, including its weather, population of cheese miners, and the potential for a union strike.\n\n## SUMMARY\n- The capital of the moon is Lunapolis.\n- The weather in Lunapolis is clear, with a high temperature of 120C and a low of -100C.\n- There are 100,000 cheese miners living in Lunapolis.\n- The cheese miners' union is likely to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='f698db7d-58cd-493c-9717-e4cc71cc6c0c'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='cce411be-43d6-478b-bc29-c5dbe1b29e04'),
              AIMessage(content='Nice –

In [5]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about fictional details related to Lunapolis, the capital of the moon, including its weather, population of cheese miners, and the potential for a union strike.

## SUMMARY
- The capital of the moon is Lunapolis.
- The weather in Lunapolis is clear, with a high temperature of 120C and a low of -100C.
- There are 100,000 cheese miners living in Lunapolis.
- The cheese miners' union is likely to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import ToolMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import RemoveMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) ->dict[str, Any] | None:
    """remove all the tool messages from the state"""
    
    messages=state["messages"]
    tool_messages= [m for m in messages if isinstance(m, ToolMessage)]
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [11]:
agent= create_agent(
    model= "gpt-5-nano",
    checkpointer= InMemorySaver(),
    middleware=[trim_messages]
)

In [12]:
response= agent.invoke(
    {
        "messages": [
            HumanMessage(content="My device won't turn on. What should I do?"),
            ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
            AIMessage(content="Is the device plugged in and turned on?"),
            HumanMessage(content="Yes, it's plugged in and turned on."),
            ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
            AIMessage(content="Is the device showing any lights or indicators?"),
            HumanMessage(content="What's the temperature of the device?")
        ]
    },
    {
        "configurable": {
            "thread_id": "1"
        }
    }
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='4d14c539-28a6-4215-9137-616ef111a785'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='be7dece8-7428-40e0-a537-2b3d026e3ab7', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='7a5a971b-25b4-4a24-90d7-f8e63fc531fe'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='5d59af88-52f1-410d-99c1-b9f6751086e1', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='f43b1a6e-cc3b-42aa-a345-d45718712535'),
              AIMessage(content='I can’t read the device’s internal temperature if it’s not pow

In [15]:
print(response["messages"][-1].content)

I can’t read the device’s internal temperature if it’s not powering on. If the device feels hot to the touch, power it off and let it cool in a well-ventilated area before trying to turn it on again.

Troubleshooting steps you can try (general):
- Safety first: If it’s hot or you smell burning, unplug it and don’t try to turn it on again until it’s cooled.
- Check power basics: try a different outlet, try a different charger or power cable (if applicable), and look for any lights on the charger or device when you plug it in.
- Force a restart: with the device off, press and hold the power button for about 15–20 seconds (some devices need 10 seconds; if there’s a small reset button, you may need a pin to press it). Then release and try turning it on normally.
- If it has a removable battery (like some laptops): power off, remove the battery, hold the power button for 15 seconds, reinsert the battery, and try again.
- Listen and look for signs: any blinking lights, sounds, or fans? Even 